# Project 2

# NYC Noise Complaints and Temperature: Exploring Daily Patterns (2019–2021)
In this project, NYC 311 Noise Complaints and NYC Weather data sets are used. 

Links to the data: 
[NYC_311 Noise](https://data.cityofnewyork.us/Social-Services/311-Noise-Complaints-from-2010-to-Present/nuav-4pki/about_data), [NYC_Weather](https://www.kaggle.com/datasets/aadimator/nyc-weather-2016-to-2022?resource=download)

The goal is to answer the following questin: Do noise complaints increase on warmer days in NYC? 

I will show visually the realtionship between the two data sets above to answer the question. 



## Data Sources

I use two different datasets:

**1. NYC 311 Noise Complaint Data (2019–2021)**  
Source: NYC Open Data  
This dataset includes detailed information about all 311 service requests.  
I focus specifically on rows where the “Complaint Type” contains the word *Noise*.  
The dataset contains timestamps, incident locations, and complaint categories.

**2. NYC Weather Data (2016–2022)**  
Source: Kaggle – “NYC Weather (2016–2022)”  
This dataset includes hourly weather observations for New York City.  
I compute a **daily average temperature** for comparability with the noise data.

Both datasets are filtered and cleaned so that they can be merged on a common key: **date**.


In [ ]:
import pandas as pd

# Load the noise complaint data
df_311 = pd.read_csv("311_Noise_Complaints.csv", low_memory=False)

print(df_311.head())
print(df_311.info())


/var/folders/k1/lny40b654c35nk73h3xmgr1h0000gn/T/ipykernel_56788/979199306.py:3: DtypeWarning: Columns (17,18,20,31) have mixed types. Specify dtype option on import or set low_memory=False.
  df_N = pd.read_csv("311_Noise_Complaints.csv")


   Unique Key            Created Date Closed Date Agency  \
0    66880639  11/19/2025 02:06:14 AM         NaN   NYPD   
1    66874880  11/19/2025 02:05:41 AM         NaN   NYPD   
2    66874869  11/19/2025 02:03:08 AM         NaN   NYPD   
3    66874875  11/19/2025 02:02:20 AM         NaN   NYPD   
4    66880648  11/19/2025 02:01:12 AM         NaN   NYPD   

                       Agency Name           Complaint Type        Descriptor  \
0  New York City Police Department      Noise - Residential  Banging/Pounding   
1  New York City Police Department  Noise - Street/Sidewalk  Loud Music/Party   
2  New York City Police Department      Noise - Residential  Banging/Pounding   
3  New York City Police Department      Noise - Residential  Banging/Pounding   
4  New York City Police Department  Noise - Street/Sidewalk  Loud Music/Party   

                Location Type  Incident Zip      Incident Address  ...  \
0  Residential Building/House       11373.0  41-18 HAMPTON STREET  ...   
1   

In [ ]:

# Load weather file
df_w = pd.read_csv("NYC_Weather_2016_2022.csv")

print(df_w.head())
print(df_w.info())

               time  temperature_2m (°C)  precipitation (mm)  rain (mm)  \
0  2016-01-01T00:00                  7.6                 0.0        0.0   
1  2016-01-01T01:00                  7.5                 0.0        0.0   
2  2016-01-01T02:00                  7.1                 0.0        0.0   
3  2016-01-01T03:00                  6.6                 0.0        0.0   
4  2016-01-01T04:00                  6.3                 0.0        0.0   

   cloudcover (%)  cloudcover_low (%)  cloudcover_mid (%)  \
0            69.0                53.0                 0.0   
1            20.0                 4.0                 0.0   
2            32.0                 3.0                 0.0   
3            35.0                 5.0                 0.0   
4            34.0                 4.0                 0.0   

   cloudcover_high (%)  windspeed_10m (km/h)  winddirection_10m (°)  
0                 72.0                  10.0                  296.0  
1                 56.0                   9

## Data Cleaning and Preparation

### Noise Data
- I converted the `Created Date` column to datetime.
- Selected only rows where the `Complaint Type` contains the word **Noise**.
- Filtered the data to include only the years **2019, 2020, and 2021**.
- Aggregated the data into **daily counts** of noise complaints.

### Weather Data
- Converted the timestamp (`time`) column to datetime.
- Extracted the date component.
- Filtered the dataset to 2019–2021 to match the noise data.
- Computed a **daily average temperature**.

### Merging the Data
Using `pd.merge`, I joined the two datasets on the `date` column.
This produced one combined dataset with:

- `date`
- `noise_count`
- `avg_temp`

This merged dataset allowed me to produce the main visualization combining both datasets.


In [ ]:
#clean and filter noise data 

# Convert to datetime
df_311['Created Date'] = pd.to_datetime(df_311['Created Date'], errors='coerce')

# Filter to noise complaints only
df_noise = df_311[df_311['Complaint Type'].str.contains("Noise", case=False, na=False)].copy()

# Filter years 2019, 2020, 2021
df_noise = df_noise[(df_noise['Created Date'].dt.year >= 2019) &
                    (df_noise['Created Date'].dt.year <= 2021)]

# Group by date
df_noise_daily = (
    df_noise.groupby(df_noise['Created Date'].dt.date)
            .size()
            .reset_index(name='noise_count')
)

# Rename date column
df_noise_daily = df_noise_daily.rename(columns={'Created Date': 'date'})
df_noise_daily['date'] = pd.to_datetime(df_noise_daily['date'])

df_noise_daily.head()


,date,noise_count
0,2019-01-01,1566
1,2019-01-02,691
2,2019-01-03,767
3,2019-01-04,924
4,2019-01-05,1172


In [ ]:
#clean and filter weather data 

# Convert time column to datetime
df_w["time"] = pd.to_datetime(df_w["time"], errors="coerce")

# Create date column (daily)
df_w["date"] = df_w["time"].dt.date
df_w["date"] = pd.to_datetime(df_w["date"])

# Filter to 2019–2021 only
df_w = df_w[(df_w["date"].dt.year >= 2019) & (df_w["date"].dt.year <= 2021)]

# Create daily average temperature
df_weather_daily = (
    df_w.groupby("date")["temperature_2m (°C)"]
        .mean()
        .reset_index(name="avg_temp")
)

df_weather_daily.head()


,date,avg_temp
0,2019-01-01,10.050000
1,2019-01-02,3.408333
2,2019-01-03,4.479167
3,2019-01-04,3.162500
4,2019-01-05,6.220833


In [ ]:
#Merge both data sets

merged = pd.merge(df_noise_daily, df_weather_daily, on="date", how="inner")
merged.head()


,date,noise_count,avg_temp
0,2019-01-01,1566,10.050000
1,2019-01-02,691,3.408333
2,2019-01-03,767,4.479167
3,2019-01-04,924,3.162500
4,2019-01-05,1172,6.220833


## Main Visualization: Daily Noise Complaints vs Temperature (2019–2021)

The figure below shows both datasets on the same timeline:

- The **blue line** shows daily noise complaints reported through NYC’s 311 system.
- The **red line** shows the daily average temperature in °C.

### Interpretation

Several clear patterns emerge:

1. **Seasonality is very strong.**  
   Noise complaints rise sharply during warmer months (May–September) and fall during colder months.

2. **The warmest periods consistently align with the highest noise levels.**  
   This pattern appears in all three years.

3. **The COVID-19 period (mid-2020) shows an unusual drop** in noise complaints even during warm months, which aligns with reduced nightlife, reduced gatherings, and lockdowns.

Overall, the visualization suggests a **positive association** between temperature and noise activity in NYC.


In [ ]:
import plotly.graph_objs as go

fig = go.Figure()

# Noise complaints line (left axis)
fig.add_trace(
    go.Scatter(
        x=merged["date"],
        y=merged["noise_count"],
        mode="lines",
        name="Daily Noise Complaints"
    )
)

# Temperature line (right axis)
fig.add_trace(
    go.Scatter(
        x=merged["date"],
        y=merged["avg_temp"],
        mode="lines",
        name="Average Temperature (°C)",
        yaxis="y2"
    )
)

fig.update_layout(
    title="NYC Daily Noise Complaints vs Average Temperature (2019–2021)",
    xaxis=dict(
        title="Date"
    ),
    yaxis=dict(
        title=dict(text="Daily Noise Complaints")
    ),
    yaxis2=dict(
        title=dict(text="Average Temperature (°C)"),
        overlaying="y",
        side="right"
    ),
    legend=dict(
        x=0.01,
        y=0.99
    ),
    width=1000,
    height=500
)

fig.show()


In [ ]:
weekly=merged.resample("W", on="date").sum().reset_index()
weekly

,date,noise_count,avg_temp
0,2019-01-06,6282,33.037500
1,2019-01-13,6225,1.229167
2,2019-01-20,6055,-6.037500
3,2019-01-27,6415,-8.433333
4,2019-02-03,6488,-30.916667
...,...,...,...
152,2021-12-05,9862,36.637500
153,2021-12-12,9966,45.325000
154,2021-12-19,9890,55.295833
155,2021-12-26,8213,20.075000




To make the graph easeir for the reader, I resampled the days into week and ploted it below



In [ ]:
fig = go.Figure()

# Noise complaints line (left axis)
fig.add_trace(
    go.Scatter(
        x=weekly["date"],
        y=weekly["noise_count"],
        mode="lines",
        name="Weekly Noise Complaints"
    )
)

# Temperature line (right axis)
fig.add_trace(
    go.Scatter(
        x=weekly["date"],
        y=weekly["avg_temp"],
        mode="lines",
        name="Average Temperature (°C)",
        yaxis="y2"
    )
)

fig.update_layout(
    title="NYC Weekly Noise Complaints vs Average Temperature (2019–2021)",
    xaxis=dict(
        title="Date"
    ),
    yaxis=dict(
        title=dict(text="Daily Noise Complaints")
    ),
    yaxis2=dict(
        title=dict(text="Average Temperature (°C)"),
        overlaying="y",
        side="right"
    ),
    legend=dict(
        x=0.01,
        y=0.99
    ),
    width=1000,
    height=500
)

fig.show()

## Supporting Visualization: Temperature vs Noise Complaints (Scatterplot)

The scatterplot below shows the direct relationship between temperature and daily noise complaints.

A regression line is included to highlight the trend.

### Interpretation

- There is a **clear upward-sloping trend**, indicating that higher temperatures are associated with more noise complaints.
- Days with temperatures above ~20°C show noticeably higher complaint counts.
- Colder days (below 0°C) rarely see high noise activity.

### Key Takeaways

1. **Temperature is strongly correlated with noise activity.**  
   Warmer days lead to more outdoor socializing, open windows, nightlife, and street activity.

2. **Behavioral patterns shifted during COVID (2020).**  
   Even warm days in 2020 experienced fewer complaints than warm days in 2019 or 2021.

3. **The relationship holds across all three years**, suggesting a robust connection between weather and noise-related behavior in NYC.


In [ ]:
import plotly.express as px

fig_scatter = px.scatter(
    merged,
    x="avg_temp",
    y="noise_count",
    trendline="ols",
    opacity=0.5,
    title="Relationship Between Temperature and Daily Noise Complaints (2019–2021)",
    labels={
        "avg_temp": "Average Temperature (°C)",
        "noise_count": "Daily Noise Complaints"
    }
)

fig_scatter.show()


## Conclusion

By combining two independent datasets, NYC daily noise complaints and NYC daily average temperature, I was able to analyze behavioral and seasonal dynamics in the city.

The combined visualization and scatter plot clearly show that:

- Noise complaints follow a seasonal pattern driven by temperature.
- Louder days are overwhelmingly clustered in warmer months.
- COVID-19 temporarily altered these patterns, demonstrating how external shocks can reshape urban behavior.

This project demonstrates the value of merging datasets to generate new insights and highlights how weather can influence urban noise patterns.
